# Построение нейронной сети

In [2]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

## Получение устройства для обучения (CPU/GPU...)

In [4]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else 'cpu'
print(f'Using {device} device')

Using mps device


## Определение класса

In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [6]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [9]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_proba = nn.Softmax(dim=1)(logits)
y_pred = pred_proba.argmax(1)
print(f'predictes class: {y_pred.item()}')

predictes class: 8


## Слои модели

In [10]:
input_image = torch.rand(3, 28, 28)
print(input_image.shape)

torch.Size([3, 28, 28])


### nn.Flatten 

Мы инициализируем nn.Flatten слой для преобразования каждого изображения 2D 28x28 в сопредельный массив из значений 784 пикселей.

In [11]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


### nn.Linear

Линейный слой это модуль, который применяет линейное преобразование на входе с использованием его накопленных весов и смещений.

In [12]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


### nn.ReLU

Нелинейные активации — это то, что создает сложные отображения между входами и выходами модели. Они применяются после линейных преобразований для введения нелинейности, помогая нейронным сетям изучать самые разнообразные закономерности.

В этой модели мы используем nn.ReLU между нашими линейными слоями, но есть и другие активации, чтобы ввести нелинейность в вашей модели.

In [13]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-3.3758e-01,  3.8833e-02,  4.8817e-01, -3.9149e-01, -6.8463e-01,
          4.0092e-04, -4.3038e-02,  3.4541e-03, -5.7562e-01, -2.8564e-01,
          1.8487e-01,  1.2979e-01, -3.9054e-01,  3.6442e-01,  3.7063e-02,
          1.4876e-01, -2.1103e-04, -5.2410e-01,  1.7147e-01,  1.3462e-01],
        [-2.1828e-01, -4.5557e-02,  5.7170e-01, -1.2835e-01, -4.1004e-01,
          3.4675e-02, -1.2304e-01,  9.4439e-02, -6.0521e-01, -2.2784e-01,
         -1.1038e-01,  4.3975e-01, -4.3169e-01,  9.0327e-02, -3.1880e-01,
         -1.4534e-02, -1.0024e-01, -7.4206e-01,  5.5915e-01,  3.5210e-01],
        [-5.4883e-01, -1.7914e-02,  4.3529e-01, -3.8063e-01, -5.2506e-01,
         -4.8417e-01, -1.1898e-01, -3.9298e-03, -2.7020e-01, -5.1648e-02,
          9.6973e-02,  6.5831e-02, -5.4195e-01,  1.0397e-01, -1.8826e-01,
          2.1133e-01,  1.8441e-01, -5.1001e-01,  1.6886e-01,  2.0901e-01]],
       grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0000e+00, 3.8833e-02, 4.8817e-01, 0.0

### nn.Sequential 

`nn.Sequential` - это контейнер. Данные передаются через все модули в том же порядке, что и определены. Вы можете использовать последовательные контейнеры для создания быстрой сети, как `seq_modules`.

In [14]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)
logits

tensor([[-0.1072,  0.2188,  0.3284,  0.1560,  0.1575, -0.1343, -0.3007, -0.1108,
         -0.1217, -0.1079],
        [-0.1120,  0.1623,  0.3557,  0.1738,  0.1543, -0.0234, -0.1563, -0.0097,
         -0.1447, -0.0777],
        [-0.1278,  0.1664,  0.2847,  0.2208,  0.1758, -0.2639, -0.2516,  0.0726,
         -0.0655,  0.0428]], grad_fn=<AddmmBackward0>)

### nn.Softmax 

Последний линейный слой нейронной сети возвращает логиты - необработанные значения в [-infty, infty] - которые передаются `nn.Softmax` модуль. Логиты масштабируются до значений [0, 1] представляет прогнозируемые вероятности модели для каждого класса. `dim` параметр указывает ось, вдоль которой значения должны суммироваться до 1.

In [16]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)
pred_probab

tensor([[0.0884, 0.1224, 0.1366, 0.1150, 0.1152, 0.0860, 0.0728, 0.0881, 0.0871,
         0.0883],
        [0.0854, 0.1124, 0.1364, 0.1137, 0.1115, 0.0933, 0.0817, 0.0946, 0.0827,
         0.0884],
        [0.0844, 0.1132, 0.1274, 0.1196, 0.1143, 0.0736, 0.0745, 0.1031, 0.0898,
         0.1001]], grad_fn=<SoftmaxBackward0>)

## Параметры модели

Многие слои внутри нейронной сети параметризованы, т.е. имеют связанные веса и смещения, которые оптимизированы во время обучения. Подкласс `nn.Module` автоматически отслеживает все поля, определенные внутри объекта модели, и делает все параметры доступными с помощью вашей модели через `parameters()` или `named_parameters()` методы.

В этом примере мы повторяем каждый параметр и печатаем его размер и предварительный просмотр его значений.

In [17]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[-0.0073,  0.0299, -0.0049,  ..., -0.0128, -0.0356,  0.0007],
        [-0.0082, -0.0114,  0.0098,  ..., -0.0257, -0.0186,  0.0241]],
       device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([ 9.1821e-05, -2.0791e-02], device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[-0.0297, -0.0425,  0.0303,  ..., -0.0432, -0.0238, -0.0127],
        [ 0.0436, -0.0129, -0.0337,  ...,  0.0371, -0.0318,  0.0254]],
       device='mps:0', grad_f